## Exercise 1

**Dataset Used:** Custom/Synthetic Array Data (numpy)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 15: Projektarbeit & Best Practices
# Niveau: Anfänger
# Aufgabe 1 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

import numpy as np
import matplotlib

import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix, classification_report

# Reproduzierbarkeit (wichtige Best Practice!)
np.random.seed(42)
tf.random.set_seed(42)

print("=== End-to-End Projekt: Bilderkennung (synthetische Daten) ===\n")
print("Aufgabe: Klassifizierung von 3 Klassen – Quadrate, Kreise, Dreiecke")

# ═══════════════════════════════════════════════════════════
# SCHRITT 1: DATENSATZ GENERIEREN
# ═══════════════════════════════════════════════════════════

IMG_SIZE  = 28   # 28×28 Pixel
N_SAMPLES = 3000  # pro Klasse
N_CLASSES = 3
CLASS_NAMES = ['Quadrat', 'Kreis', 'Dreieck']

def draw_square(img_size=28):
    """Zeichnet ein zufällig positioniertes Quadrat."""
    img = np.zeros((img_size, img_size), dtype=np.float32)
    size  = np.random.randint(8, img_size // 2)
    x0 = np.random.randint(0, img_size - size)
    y0 = np.random.randint(0, img_size - size)
    img[y0:y0+size, x0:x0+size] = 1.0
    return img

def draw_circle(img_size=28):
    """Zeichnet einen zufällig positionierten Kreis."""
    img = np.zeros((img_size, img_size), dtype=np.float32)
    r   = np.random.randint(4, img_size // 4)
    cx  = np.random.randint(r, img_size - r)
    cy  = np.random.randint(r, img_size - r)
    Y, X = np.ogrid[:img_size, :img_size]
    dist = (X - cx)**2 + (Y - cy)**2
    img[dist <= r**2] = 1.0
    return img

def draw_triangle(img_size=28):
    """Zeichnet ein zufällig positioniertes Dreieck."""
    img = np.zeros((img_size, img_size), dtype=np.float32)
    size = np.random.randint(8, img_size // 2)
    x0   = np.random.randint(0, img_size - size)
    y0   = np.random.randint(0, img_size - size)
    for row in range(size):
        width = int((row + 1) * size / size)
        left  = x0 + (size - width) // 2
        right = left + width
        y     = y0 + row
        if 0 <= y < img_size:
            img[y, max(0, left):min(img_size, right)] = 1.0
    return img

print("Generiere synthetischen Datensatz...")
draw_fns = [draw_square, draw_circle, draw_triangle]
X_data = []
y_data = []

for class_idx, draw_fn in enumerate(draw_fns):
    for _ in range(N_SAMPLES):
        img = draw_fn(IMG_SIZE)
        # Kleines Rauschen hinzufügen
        img = img + 0.1 * np.random.randn(*img.shape)
        img = np.clip(img, 0, 1)
        X_data.append(img.flatten())
        y_data.append(class_idx)

X_data = np.array(X_data, dtype=np.float32)
y_data = np.array(y_data, dtype=np.int32)

print(f"Datensatz: {X_data.shape[0]} Samples, {N_CLASSES} Klassen")

# ═══════════════════════════════════════════════════════════
# SCHRITT 2: TRAIN/VAL/TEST SPLIT
# ═══════════════════════════════════════════════════════════
from sklearn.model_selection import train_test_split

# Erst 80/20 split, dann Val aus Train
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X_data, y_data, test_size=0.2, random_state=42, stratify=y_data)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=42, stratify=y_train_full)

print(f"Train: {X_train.shape[0]}, Val: {X_val.shape[0]}, Test: {X_test.shape[0]}")

# ═══════════════════════════════════════════════════════════
# SCHRITT 3: CNN MODELL AUFBAUEN
# ═══════════════════════════════════════════════════════════
model = keras.Sequential([
    # Reshape für Conv2D: (batch, 784) → (batch, 28, 28, 1)
    layers.Reshape((IMG_SIZE, IMG_SIZE, 1), input_shape=(784,)),

    # Convolutional Block 1
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),

    # Convolutional Block 2
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),

    # Klassifikationskopf
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(N_CLASSES, activation='softmax')
], name='shape_classifier_cnn')

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

# ═══════════════════════════════════════════════════════════
# SCHRITT 4: TRAINING MIT EARLY STOPPING
# ═══════════════════════════════════════════════════════════
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=8, restore_best_weights=True,
        verbose=1
    )
]

print("\nTrainiere CNN...")
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)

# ═══════════════════════════════════════════════════════════
# SCHRITT 5: EVALUATION
# ═══════════════════════════════════════════════════════════
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"\nTest-Accuracy: {test_acc:.4f}")
print(f"Test-Loss:     {test_loss:.4f}")

y_pred_probs = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

print("\nKlassifikationsbericht:")
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

# ═══════════════════════════════════════════════════════════
# SCHRITT 6: CONFUSION MATRIX + VISUALISIERUNG
# ═══════════════════════════════════════════════════════════
cm = confusion_matrix(y_test, y_pred)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('End-to-End Projekt: Formenerkennung CNN', fontsize=14)

# Confusion Matrix
ax = axes[0, 0]
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(N_CLASSES)); ax.set_xticklabels(CLASS_NAMES)
ax.set_yticks(range(N_CLASSES)); ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel('Vorhersage'); ax.set_ylabel('Wahr')
ax.set_title(f'Confusion Matrix (Acc: {test_acc:.3f})')
for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                 color='white' if cm[i, j] > cm.max()/2 else 'black', fontsize=12)
plt.colorbar(im, ax=ax)

# Lernkurven
ax2 = axes[0, 1]
ax2.plot(history.history['accuracy'],     label='Train-Acc')
ax2.plot(history.history['val_accuracy'], label='Val-Acc')
ax2.set_title('Lernkurven')
ax2.set_xlabel('Epoche'); ax2.set_ylabel('Accuracy')
ax2.legend(); ax2.grid(True)

# Beispielbilder
ax3 = axes[1, 0]
n_show = 9
for i in range(n_show):
    ax3.imshow(X_test[i].reshape(IMG_SIZE, IMG_SIZE), cmap='gray',
                extent=[i*30, (i+1)*30, 0, 30], aspect='auto')
ax3.axis('off')
ax3.set_title('Beispiel-Testbilder')

# Klassen-Accuracy
ax4 = axes[1, 1]
per_class_acc = cm.diagonal() / cm.sum(axis=1)
ax4.bar(CLASS_NAMES, per_class_acc, color=['red', 'green', 'blue'], alpha=0.8)
ax4.set_title('Accuracy pro Klasse')
ax4.set_ylabel('Accuracy'); ax4.set_ylim(0, 1.1); ax4.grid(True, axis='y')
for i, acc in enumerate(per_class_acc):
    ax4.text(i, acc + 0.01, f'{acc:.3f}', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('A15_1_bilderkennung_projekt.png', dpi=100)
plt.show()

print("\nProjekt abgeschlossen!")
print(f"Finale Test-Accuracy: {test_acc:.4f}")


## Exercise 2

**Dataset Used:** Custom/Synthetic Array Data (numpy)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 15: Projektarbeit & Best Practices
# Niveau: Anfänger
# Aufgabe 2 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

import numpy as np
import matplotlib

import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Reproduzierbarkeit
np.random.seed(42)
tf.random.set_seed(42)

print("=== End-to-End Projekt: Zeitreihenprognose mit LSTM ===\n")

# ═══════════════════════════════════════════════════════════
# SCHRITT 1: SYNTHETISCHE ZEITREIHE GENERIEREN
# ═══════════════════════════════════════════════════════════

def generate_time_series(n_steps=1000, noise_std=0.1):
    """
    Generiert eine synthetische Zeitreihe:
    - Sinus-Welle (Hauptfrequenz)
    - Oberton (halbe Frequenz)
    - Linearer Trend
    - Gausssches Rauschen
    """
    t = np.linspace(0, 10 * np.pi, n_steps)

    # Sinus-Komponenten
    sine1 = np.sin(t)
    sine2 = 0.3 * np.sin(2 * t + np.pi / 4)

    # Trend
    trend = 0.05 * t / t.max()

    # Gesamte Zeitreihe
    signal = sine1 + sine2 + trend + noise_std * np.random.randn(n_steps)

    return signal.astype(np.float32), t

N_TOTAL = 1200
signal, t = generate_time_series(N_TOTAL)
print(f"Zeitreihe: {N_TOTAL} Datenpunkte")
print(f"Signal-Bereich: [{signal.min():.3f}, {signal.max():.3f}]")

# ═══════════════════════════════════════════════════════════
# SCHRITT 2: FENSTERDATENSATZ ERSTELLEN
# ═══════════════════════════════════════════════════════════

WINDOW_SIZE = 50    # Eingabe: 50 Zeitschritte
FORECAST    = 1     # Vorhersage: nächster Zeitschritt

def create_windowed_dataset(series, window_size, forecast_steps=1):
    """
    Konvertiert Zeitreihe in Fenster-Format.
    X: [t-window_size, ..., t-1]
    y: [t, ..., t+forecast_steps-1]
    """
    X, y = [], []
    for i in range(len(series) - window_size - forecast_steps + 1):
        X.append(series[i : i + window_size])
        y.append(series[i + window_size : i + window_size + forecast_steps])
    return np.array(X), np.array(y)

X, y = create_windowed_dataset(signal, WINDOW_SIZE, FORECAST)

# LSTM benötigt Shape (batch, timesteps, features)
X = X.reshape(X.shape[0], X.shape[1], 1)
print(f"X Shape: {X.shape}  (Samples, Timesteps, Features)")
print(f"y Shape: {y.shape}")

# Train/Val/Test Split (zeitbasiert – kein zufälliger Split!)
n = len(X)
train_end = int(n * 0.7)
val_end   = int(n * 0.85)

X_train, y_train = X[:train_end],       y[:train_end]
X_val,   y_val   = X[train_end:val_end], y[train_end:val_end]
X_test,  y_test  = X[val_end:],          y[val_end:]

print(f"\nSplit (zeitbasiert): Train={len(X_train)}, Val={len(X_val)}, Test={len(X_test)}")

# ═══════════════════════════════════════════════════════════
# SCHRITT 3: LSTM MODELL AUFBAUEN
# ═══════════════════════════════════════════════════════════

model = keras.Sequential([
    layers.LSTM(64, return_sequences=True, input_shape=(WINDOW_SIZE, 1)),
    layers.LSTM(32, return_sequences=False),
    layers.Dense(16, activation='relu'),
    layers.Dense(FORECAST)  # Ausgabe: Vorhersagewert(e)
], name='lstm_forecast')

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)
model.summary()

# ═══════════════════════════════════════════════════════════
# SCHRITT 4: TRAINING
# ═══════════════════════════════════════════════════════════
callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=15,
                                    restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                        patience=5, verbose=1)
]

print("\nTrainiere LSTM...")
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=80,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)

# ═══════════════════════════════════════════════════════════
# SCHRITT 5: VORHERSAGE DER NÄCHSTEN 20 SCHRITTE
# ═══════════════════════════════════════════════════════════

def forecast_multistep(model, last_window, n_steps=20, window_size=WINDOW_SIZE):
    """
    Iterative Mehrschritt-Vorhersage:
    Benutzt jeweils die letzte Vorhersage als neuen Input.
    """
    current_window = last_window.copy().reshape(1, window_size, 1)
    forecasts = []

    for _ in range(n_steps):
        next_pred = model.predict(current_window, verbose=0)[0, 0]
        forecasts.append(next_pred)
        # Fenster um 1 vorrücken
        current_window = np.roll(current_window, -1, axis=1)
        current_window[0, -1, 0] = next_pred

    return np.array(forecasts)

# Letztes verfügbares Fenster
last_window = X_test[-1].reshape(WINDOW_SIZE)
forecast_20 = forecast_multistep(model, last_window, n_steps=20)

# Referenz: Echte zukünftige Werte (falls vorhanden)
last_signal_idx = val_end + len(X_test) + WINDOW_SIZE
future_signal   = signal[last_signal_idx : last_signal_idx + 20] if last_signal_idx + 20 <= N_TOTAL else None

# ═══════════════════════════════════════════════════════════
# SCHRITT 6: EVALUIERUNG
# ═══════════════════════════════════════════════════════════
test_preds   = model.predict(X_test, verbose=0).flatten()
test_actuals = y_test.flatten()
test_mse = np.mean((test_preds - test_actuals) ** 2)
test_mae = np.mean(np.abs(test_preds - test_actuals))

print(f"\nTest MSE: {test_mse:.5f}")
print(f"Test MAE: {test_mae:.5f}")

# Konfidenzintervall: ±Std der Residuen
residuals  = test_preds - test_actuals
pred_std   = residuals.std()
print(f"Residuen-Std (Konfidenzintervall ±): {pred_std:.5f}")

# ═══════════════════════════════════════════════════════════
# SCHRITT 7: VISUALISIERUNG
# ═══════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('LSTM Zeitreihenprognose', fontsize=14)

# Gesamte Zeitreihe + Train/Val/Test Aufteilung
ax = axes[0, 0]
ax.plot(signal[:train_end + WINDOW_SIZE], color='blue', alpha=0.7, label='Training')
ax.plot(range(train_end + WINDOW_SIZE, val_end + WINDOW_SIZE),
         signal[train_end + WINDOW_SIZE : val_end + WINDOW_SIZE],
         color='orange', alpha=0.7, label='Validierung')
ax.plot(range(val_end + WINDOW_SIZE, N_TOTAL),
         signal[val_end + WINDOW_SIZE:], color='green', alpha=0.7, label='Test')
ax.set_title('Zeitreihe mit Train/Val/Test Split')
ax.set_xlabel('Zeitschritt'); ax.set_ylabel('Wert')
ax.legend(); ax.grid(True)

# Vorhersage auf Test-Segment
ax2 = axes[0, 1]
n_show = min(200, len(test_actuals))
ax2.plot(test_actuals[:n_show], label='Tatsächlich', color='blue')
ax2.plot(test_preds[:n_show],   label='Vorhergesagt', color='red', alpha=0.7)
ax2.fill_between(range(n_show),
                  test_preds[:n_show] - pred_std,
                  test_preds[:n_show] + pred_std,
                  alpha=0.2, color='red', label=f'±Std ({pred_std:.3f})')
ax2.set_title(f'Vorhersage auf Testdaten (MAE={test_mae:.4f})')
ax2.set_xlabel('Zeitschritt'); ax2.set_ylabel('Wert')
ax2.legend(); ax2.grid(True)

# 20-Schritt Vorwärtsprognose
ax3 = axes[1, 0]
context = last_window[-30:]   # Letzte 30 bekannte Werte
ax3.plot(range(len(context)), context, color='blue', label='Bekannt')
ax3.plot(range(len(context), len(context)+20), forecast_20,
          color='red', marker='o', markersize=4, label='Prognose')
ax3.fill_between(range(len(context), len(context)+20),
                  forecast_20 - pred_std, forecast_20 + pred_std,
                  alpha=0.2, color='red', label=f'±Std Konfidenz')
if future_signal is not None and len(future_signal) > 0:
    ax3.plot(range(len(context), len(context) + len(future_signal)),
              future_signal, color='green', linestyle='--', label='Tatsächlich')
ax3.set_title('20-Schritt Vorwärtsprognose')
ax3.set_xlabel('Zeitschritt'); ax3.set_ylabel('Wert')
ax3.legend(); ax3.grid(True)

# Training-Verlauf
ax4 = axes[1, 1]
ax4.plot(history.history['loss'],     label='Train-MSE')
ax4.plot(history.history['val_loss'], label='Val-MSE')
ax4.set_title('Training-Verlauf')
ax4.set_xlabel('Epoche'); ax4.set_ylabel('MSE')
ax4.legend(); ax4.grid(True)

plt.tight_layout()
plt.savefig('A15_2_zeitreihen_prognose.png', dpi=100)
plt.show()

print("\nZeitreihen-Projekt abgeschlossen!")


## Exercise 3

**Dataset Used:** MNIST (keras.datasets)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 15: Projektarbeit & Best Practices
# Niveau: Anfänger
# Aufgabe 3 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

import numpy as np
import matplotlib

import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import time

print("=== Best Practices Checkliste als Code ===\n")

# ─────────────────────────────────────────────
# BEST PRACTICE 1: REPRODUZIERBARKEIT
# Alle Zufalls-Seeds setzen!
# ─────────────────────────────────────────────
import os
import random

def set_all_seeds(seed=42):
    """Setzt alle relevanten Zufalls-Seeds für Reproduzierbarkeit."""
    np.random.seed(seed)
    tf.random.set_seed(seed)
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    print(f"✓ Alle Seeds gesetzt: {seed}")

set_all_seeds(42)

# ─────────────────────────────────────────────
# BEST PRACTICE 2: FUNKTIONEN STATT COPY-PASTE
# Jede Aufgabe in eine Funktion auslagern
# ─────────────────────────────────────────────

def load_and_preprocess_data():
    """Lädt und normalisiert MNIST-Daten."""
    (x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
    x_train = x_train.astype('float32') / 255.0
    x_test  = x_test.astype('float32') / 255.0
    x_train = x_train.reshape(-1, 784)
    x_test  = x_test.reshape(-1, 784)
    print(f"✓ Daten geladen: Train={x_train.shape}, Test={x_test.shape}")
    return x_train, y_train, x_test, y_test

def build_model(hidden_units, activation='relu', dropout=0.0, name='model'):
    """
    Modell-Fabrik-Funktion.

    Parameter:
        hidden_units (list): Größe jeder versteckten Schicht
        activation (str): Aktivierungsfunktion
        dropout (float): Dropout-Rate
        name (str): Modell-Name

    Returns:
        Kompiliertes Keras-Modell
    """
    model_layers = [layers.Dense(hidden_units[0], activation=activation, input_shape=(784,))]
    if dropout > 0:
        model_layers.append(layers.Dropout(dropout))
    for units in hidden_units[1:]:
        model_layers.append(layers.Dense(units, activation=activation))
        if dropout > 0:
            model_layers.append(layers.Dropout(dropout / 2))
    model_layers.append(layers.Dense(10, activation='softmax'))

    model = keras.Sequential(model_layers, name=name)
    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

def train_model(model, x_train, y_train, epochs=10, batch_size=256, val_split=0.1,
                 verbose=0):
    """
    Trainiert ein Modell mit Standard-Callbacks.

    Returns:
        history Objekt
    """
    t_start = time.time()
    callbacks = [
        keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)
    ]
    history = model.fit(
        x_train, y_train,
        epochs=epochs, batch_size=batch_size,
        validation_split=val_split,
        callbacks=callbacks,
        verbose=verbose
    )
    elapsed = time.time() - t_start
    best_val_acc = max(history.history['val_accuracy'])
    print(f"  ✓ Training ({model.name}): {len(history.history['loss'])} Epochen, "
          f"beste Val-Acc={best_val_acc:.4f}, Zeit={elapsed:.1f}s")
    return history

def evaluate_model(model, x_test, y_test):
    """Evaluiert auf Test-Set und gibt Metriken zurück."""
    loss, acc = model.evaluate(x_test, y_test, verbose=0)
    print(f"  ✓ Test-Acc ({model.name}): {acc:.4f}")
    return {'loss': loss, 'accuracy': acc}

def compare_models(results_list):
    """Druckt Vergleichstabelle aller Modelle."""
    print("\n" + "="*60)
    print("MODELL-VERGLEICH")
    print("="*60)
    print(f"{'Modell':25} | {'Val-Acc':8} | {'Test-Acc':9} | {'Epochen':7}")
    print("-"*60)
    sorted_r = sorted(results_list, key=lambda x: x['test_acc'], reverse=True)
    for r in sorted_r:
        print(f"{r['name']:25} | {r['val_acc']:8.4f} | {r['test_acc']:9.4f} | {r['epochs']:7d}")
    return sorted_r

# ─────────────────────────────────────────────
# BEST PRACTICE 3: EXPERIMENT TRACKING
# ─────────────────────────────────────────────
class SimpleTracker:
    """Einfaches Experiment-Tracking ohne externe Libraries."""
    def __init__(self):
        self.experiments = []

    def log(self, name, config, metrics):
        self.experiments.append({
            'name': name, 'config': config,
            'metrics': metrics, 'timestamp': time.strftime('%H:%M:%S')
        })

    def print_summary(self):
        print("\n" + "="*60)
        print("EXPERIMENT-TRACKING LOG")
        print("="*60)
        for exp in self.experiments:
            print(f"[{exp['timestamp']}] {exp['name']}")
            print(f"  Config:  {exp['config']}")
            print(f"  Metrics: {exp['metrics']}")

# ─────────────────────────────────────────────
# HAUPTPROGRAMM
# ─────────────────────────────────────────────
tracker = SimpleTracker()

print("=== Schritt 1: Daten laden ===")
x_train, y_train, x_test, y_test = load_and_preprocess_data()

print("\n=== Schritt 2: Drei Modelle trainieren und vergleichen ===")

model_configs = [
    {'name': 'Klein',   'units': [64, 32],       'dropout': 0.0},
    {'name': 'Mittel',  'units': [128, 64],      'dropout': 0.2},
    {'name': 'Groß',    'units': [256, 128, 64], 'dropout': 0.3},
]

all_results = []
all_histories = []

for cfg in model_configs:
    print(f"\n--- Modell: {cfg['name']} ---")

    # Seed vor jedem Modell zurücksetzen für Vergleichbarkeit
    set_all_seeds(42)

    model = build_model(cfg['units'], dropout=cfg['dropout'], name=cfg['name'])
    hist  = train_model(model, x_train, y_train, epochs=20, verbose=0)
    eval_r = evaluate_model(model, x_test, y_test)

    best_val_acc = max(hist.history['val_accuracy'])
    epochs_run   = len(hist.history['loss'])

    result = {
        'name': cfg['name'], 'config': cfg,
        'val_acc': best_val_acc, 'test_acc': eval_r['accuracy'],
        'epochs': epochs_run, 'model': model, 'history': hist
    }
    all_results.append(result)
    all_histories.append(hist)

    # Tracking
    tracker.log(cfg['name'], cfg, {
        'val_acc': best_val_acc, 'test_acc': eval_r['accuracy'],
        'epochs': epochs_run
    })

# ─────────────────────────────────────────────
# BEST PRACTICE 4: ERGEBNISSE PROTOKOLLIEREN
# ─────────────────────────────────────────────
sorted_results = compare_models(all_results)
tracker.print_summary()

best_result = sorted_results[0]
print(f"\n→ Bestes Modell: {best_result['name']} (Test-Acc: {best_result['test_acc']:.4f})")

# ─────────────────────────────────────────────
# BEST PRACTICE 5: VISUALISIERUNG
# ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Best Practices: 3 Modelle im Vergleich', fontsize=13)

colors = ['blue', 'orange', 'green']

# Lernkurven
ax = axes[0]
for r, color in zip(all_results, colors):
    ax.plot(r['history'].history['val_accuracy'], color=color, label=r['name'])
ax.set_title('Val-Accuracy (Lernkurven)')
ax.set_xlabel('Epoche'); ax.set_ylabel('Val-Accuracy')
ax.legend(); ax.grid(True)

# Test-Accuracy Balken
ax2 = axes[1]
names = [r['name'] for r in all_results]
test_accs = [r['test_acc'] for r in all_results]
ax2.bar(names, test_accs, color=colors, alpha=0.8)
ax2.set_title('Finale Test-Accuracy')
ax2.set_ylabel('Accuracy'); ax2.set_ylim(0.9, 1.01); ax2.grid(True, axis='y')
for i, acc in enumerate(test_accs):
    ax2.text(i, acc + 0.001, f'{acc:.4f}', ha='center', fontsize=10)

# Epochen bis Early Stopping
ax3 = axes[2]
epochs_run = [r['epochs'] for r in all_results]
ax3.bar(names, epochs_run, color=colors, alpha=0.8)
ax3.set_title('Epochen bis Early Stopping')
ax3.set_ylabel('Epochen'); ax3.grid(True, axis='y')

plt.tight_layout()
plt.savefig('A15_3_best_practices.png', dpi=100)
plt.show()

print("\n=== Best Practices Zusammenfassung ===")
best_practices = [
    "1. REPRODUZIERBARKEIT: Alle Seeds setzen (numpy, tensorflow, python, os)",
    "2. FUNKTIONEN: Jede Aufgabe in klare, benannte Funktionen auslagern",
    "3. DOCSTRINGS: Jede Funktion mit Beschreibung und Typen dokumentieren",
    "4. EXPERIMENT TRACKING: Alle Ergebnisse systematisch protokollieren",
    "5. EARLY STOPPING: Overfitting verhindern, Rechenzeit sparen",
    "6. VERGLEICHE: Mehrere Konfigurationen systematisch testen",
    "7. VISUALISIERUNG: Ergebnisse immer grafisch darstellen",
    "8. SAUBERER CODE: Konstanten oben definieren, keine Magic Numbers",
]
for bp in best_practices:
    print(f"  ✓ {bp}")
